# Эксперимент: выбор инструмента и петли

Ноутбук запускает эксперимент локально, без API и ключей. Основная модель: Qwen3-4B из `D:\qwen-spec\models\Qwen3-4B`; путь можно переопределить аргументом `--model-path`. Видеокарта с 6 ГБ VRAM может потребовать выгрузки части модели в оперативную память, поэтому генерация может идти медленно. Эмбеддинг-модель `sentence-transformers/all-MiniLM-L6-v2` также загружается только из локального кэша.

Установите зависимости из `experiment/requirements-experiment.txt`. Полный прогон части A делает 720 генераций, часть B — до 2160 генераций (180 эпизодов × максимум 12 шагов). Запускайте их отдельно и дождитесь завершения каждой части.

## Протокол

- 100 инструментов: 4 целевых, 24 близких по смыслу отвлекающих, 72 из других доменов.
- 30 запросов с gold tool, по 7–9 на каждый целевой инструмент.
- N ∈ {5, 20, 50, 100}; во всех срезах остаются 4 целевых инструмента, остальные выбираются детерминированно.
- A показывает все N описаний; B ранжирует текущий срез по косинусному сходству эмбеддингов и показывает top-5.
- 3 повтора на запрос, N и вариант. Неуспешные и незавершённые попытки остаются в знаменателе и дают 0. Bootstrap ресэмплирует query ID (10 000 повторов).
- Эмбеддинг: `sentence-transformers/all-MiniLM-L6-v2` из локального кэша; запросы и описания на английском, mean pooling + L2-нормализация.
- Генератор: локальная Qwen3-4B; модельный ответ и фактические входные/выходные токены сохраняются в JSONL.

In [ ]:
from pathlib import Path
import os, json
ROOT = Path.cwd()
if ROOT.name == 'experiment': ROOT = ROOT.parent
os.chdir(ROOT)
print('Project:', ROOT)
tools = json.loads((ROOT/'experiment/data/tools.json').read_text(encoding='utf-8'))
queries = json.loads((ROOT/'experiment/data/queries.json').read_text(encoding='utf-8'))
print('Tools:', len(tools), 'near distractors:', sum(t['kind']=='near_distractor' for t in tools), 'queries:', len(queries))
print('Local model:', r'D:\qwen-spec\models\Qwen3-4B')

## Часть A — выбор инструмента

Сначала проверьте план без генерации. Полный запуск пишет каждую попытку в `experiment/logs/selection_local_<UTC>.jsonl`; анализатор читает только эти локальные логи и не смешивает их со старыми логами.

In [ ]:
!python experiment/build_dataset.py
!python experiment/run_selection.py --dry-run

In [ ]:
# Полный локальный прогон (720 генераций).
!python experiment/run_selection.py --model-path "D:\qwen-spec\models\Qwen3-4B"

In [ ]:
!python experiment/analyze_selection.py
summary = json.loads((ROOT/'experiment/results/selection_summary.json').read_text(encoding='utf-8'))
summary

## Часть B — петли

Три условия: нестабильный инструмент без guard; тот же инструмент с LoopGuard (3 одинаковых вызова, окно 6, также A-B-A-B); подсказка при пустом ответе без guard. Для каждого условия 20 задач × 3 повтора. Сырые ответы и фактические токены сохраняются в `loops_local_<UTC>.jsonl`.

In [ ]:
# Полный локальный прогон (до 2160 генераций в худшем случае).
!python experiment/run_loops.py --model-path "D:\qwen-spec\models\Qwen3-4B"

In [ ]:
!python experiment/analyze_loops.py
json.loads((ROOT/'experiment/results/loop_summary.json').read_text(encoding='utf-8'))